In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
change_loc=pd.read_csv('results/ST_location_simulate.txt',sep='\t',header=0,index_col=0)
df_st=pd.read_csv('data/st_count_matrix_MB.txt',sep='\t',header=0,index_col=0)


In [4]:
df_st.shape

(32285, 2695)

In [6]:
data.shape

(32287, 2695)

In [ ]:
## AUC

In [24]:
label_simulate=pd.read_csv('label_simulate.txt',sep='\t',header=0,index_col=0)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

X = df_st.T  


scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
pca = PCA(n_components=15)
pca_result = pca.fit_transform(X_scaled)  

pca_result_T = pca_result.T  

pca_df = pd.DataFrame(pca_result_T, 
                      index=[f'PC{i+1}' for i in range(15)], 
                      columns=df_st.columns)  

In [ ]:
data = np.vstack([pca_df, change_loc.T])  # shape: (n_features, n_samples)
X = data.T   
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

model = IsolationForest(
    contamination=0.05,
    random_state=42,
    n_estimators=100
)
model.fit(X_scaled)

labels = model.predict(X_scaled)
scores = -model.decision_function(X_scaled) 

results = pd.DataFrame({
    'sample': range(X.shape[0]),
    'label': labels,
    'score': scores
})
anomalies = results[results['label'] == -1]
print(f"共 {len(anomalies)} 个异常样本:")
print(anomalies)

from sklearn.decomposition import PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
plt.figure(figsize=(8,6))
plt.scatter(X_pca[labels==1, 0], X_pca[labels==1, 1], c='blue', label='Normal')
plt.scatter(X_pca[labels==-1, 0], X_pca[labels==-1, 1], c='red', label='Anomaly')
plt.legend()
plt.title('Isolation Forest Anomaly Detection')
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
results0=results.copy()

scores =  results0['score'].values 
labels = label_simulate['Abberant'].values                        

fpr, tpr, thresholds_roc = roc_curve(labels, scores)
auroc = auc(fpr, tpr)

precision, recall, thresholds_pr = precision_recall_curve(labels, scores)
# average_precision_score 
aupr = average_precision_score(labels, scores)
print(f"AUROC (ROC 曲线下面积): {auroc:.4f}")
print(f"AUPR  (PR 曲线下面积): {aupr:.4f}")


plt.figure(figsize=(10, 4))
# ROC
plt.subplot(1, 2, 1)
plt.plot(fpr, tpr, label=f'ROC (AUC = {auroc:.3f})')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate (FPR)')
plt.ylabel('True Positive Rate (TPR)')
plt.title('ROC Curve')
plt.legend()
# PR 
plt.subplot(1, 2, 2)
plt.plot(recall, precision, label=f'PR (AUPR = {aupr:.3f})')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
df_roc = pd.DataFrame({'fpr': fpr, 'tpr': tpr, 'threshold': thresholds_roc})
df_roc.to_csv('metrics/trandition_PCA_ROC.txt', sep='\t', index=False)
precision_adj = precision[:-1]
recall_adj = recall[:-1]
df_pr = pd.DataFrame({'precision': precision_adj, 'recall': recall_adj, 'threshold': thresholds_pr})
df_pr.to_csv('metrics/trandition_PCA_PR.txt', sep='\t', index=False)